# Volatility project dataset split

Notebook to split the full panel by date..

No sorting. No randomization. Just date filters.


In [6]:
import os
import pandas as pd

In [9]:
# File names / settings

full_data_path = "/content/drive/MyDrive/Colab Notebooks/vol_dataset_with_6_baselines_0030326_1601.csv"
date_col = "date"

train_out = "vol_dataset_train_20150102_20221230.csv"
val_out = "vol_dataset_validation_20230103_20241231.csv"
test_out = "vol_dataset_test_20250102_20251231.csv"
summary_out = "dataset_split_summary.csv"

# Date windows
train_start = pd.Timestamp("2015-01-02")
train_end   = pd.Timestamp("2022-12-30")

val_start = pd.Timestamp("2023-01-03")
val_end   = pd.Timestamp("2024-12-31")

test_start = pd.Timestamp("2025-01-02")
test_end   = pd.Timestamp("2025-12-31")

In [11]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(full_data_path, low_memory=False)

if date_col not in df.columns:
    raise ValueError(f"Column '{date_col}' not found.")

df[date_col] = pd.to_datetime(df[date_col], errors="raise")

# Keep original row positions so we can check that filtering did not reorder anything
df["original_row_order"] = range(len(df))

print("Full dataset shape:", df.shape)
print("Date range in full file:", df[date_col].min(), "to", df[date_col].max())

Mounted at /content/drive
Full dataset shape: (96525, 45)
Date range in full file: 2014-12-31 00:00:00 to 2025-12-31 00:00:00


In [12]:
# Chronological split
# No sorting
# No randomization
# Just date filters

train_df = df.loc[(df[date_col] >= train_start) & (df[date_col] <= train_end)].copy()
val_df   = df.loc[(df[date_col] >= val_start) & (df[date_col] <= val_end)].copy()
test_df  = df.loc[(df[date_col] >= test_start) & (df[date_col] <= test_end)].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
print("Total split rows:", len(train_df) + len(val_df) + len(test_df))
print("Rows in full file:", len(df))

Train rows: 69422
Validation rows: 18072
Test rows: 9000
Total split rows: 96494
Rows in full file: 96525


In [13]:
# Basic checks

print("Train dates:", train_df[date_col].min(), "to", train_df[date_col].max())
print("Validation dates:", val_df[date_col].min(), "to", val_df[date_col].max())
print("Test dates:", test_df[date_col].min(), "to", test_df[date_col].max())

if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise ValueError("One of the splits is empty.")

if train_df[date_col].max() >= val_df[date_col].min():
    raise ValueError("Train and validation overlap.")

if val_df[date_col].max() >= test_df[date_col].min():
    raise ValueError("Validation and test overlap.")

if not train_df["original_row_order"].is_monotonic_increasing:
    raise ValueError("Train rows were reordered.")
if not val_df["original_row_order"].is_monotonic_increasing:
    raise ValueError("Validation rows were reordered.")
if not test_df["original_row_order"].is_monotonic_increasing:
    raise ValueError("Test rows were reordered.")

print("Date and row-order checks passed.")

Train dates: 2015-01-02 00:00:00 to 2022-12-30 00:00:00
Validation dates: 2023-01-03 00:00:00 to 2024-12-31 00:00:00
Test dates: 2025-01-02 00:00:00 to 2025-12-31 00:00:00
Date and row-order checks passed.


In [14]:
# Save the 3 split files

train_save = train_df.drop(columns=["original_row_order"])
val_save   = val_df.drop(columns=["original_row_order"])
test_save  = test_df.drop(columns=["original_row_order"])

train_save.to_csv(train_out, index=False)
val_save.to_csv(val_out, index=False)
test_save.to_csv(test_out, index=False)

print("Saved:", train_out)
print("Saved:", val_out)
print("Saved:", test_out)

Saved: vol_dataset_train_20150102_20221230.csv
Saved: vol_dataset_validation_20230103_20241231.csv
Saved: vol_dataset_test_20250102_20251231.csv


In [15]:
# Build and save the summary table

def mdy(ts):
    return f"{ts.month}/{ts.day}/{ts.year}"

total_rows = len(df)

summary_df = pd.DataFrame({
    "split": ["training", "validation", "test", "total"],
    "start_date": [
        mdy(train_save[date_col].min()),
        mdy(val_save[date_col].min()),
        mdy(test_save[date_col].min()),
        mdy(df[date_col].min())
    ],
    "end_date": [
        mdy(train_save[date_col].max()),
        mdy(val_save[date_col].max()),
        mdy(test_save[date_col].max()),
        mdy(df[date_col].max())
    ],
    "rows": [
        len(train_save),
        len(val_save),
        len(test_save),
        total_rows
    ]
})

summary_df["percent_of_total"] = (summary_df["rows"] / total_rows * 100).round(4)
summary_df.to_csv(summary_out, index=False)

print(summary_df)
print("Saved:", summary_out)

        split  start_date    end_date   rows  percent_of_total
0    training    1/2/2015  12/30/2022  69422           71.9213
1  validation    1/3/2023  12/31/2024  18072           18.7226
2        test    1/2/2025  12/31/2025   9000            9.3240
3       total  12/31/2014  12/31/2025  96525          100.0000
Saved: dataset_split_summary.csv
